# Section 9 — Classification

**Input files (מהפרפרוססינג):**
- `X_train_preprocessed.csv` — features אימון
- `y_train.csv` — תוויות אימון
- `X_test_preprocessed.csv` — features טסט

**שלבים:**
1. טעינת נתונים
2. בדיקת התפלגות המחלקות
3. חלוקה ל-Train / Validation
4. מודלים — Logistic Regression, Decision Tree, Random Forest, Naive Bayes, SVM
5. 10-Fold Stratified Cross-Validation
6. Confusion Matrices + ROC Curves
7. Hyperparameter Tuning (GridSearchCV)
8. טיפול בחוסר איזון — SMOTE, ADASYN, RandomUnderSampler
9. Cost-Sensitive Learning — class_weight + threshold
10. הערכה סופית + תחזיות על סט הטסט

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, StratifiedKFold,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
)

try:
    from imblearn.over_sampling import SMOTE, ADASYN
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN_AVAILABLE = True
    print('imbalanced-learn loaded.')
except ImportError:
    IMBLEARN_AVAILABLE = False
    print('imbalanced-learn not installed.')
    print('To install:  pip install imbalanced-learn')

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
try:
    sns.set_theme(style='whitegrid', palette='muted')
except AttributeError:
    sns.set(style='whitegrid', palette='muted')

## Step 9.1 — Load Preprocessed Data

טוענים את קבצי ה-CSV שנוצרו בסוף שלב הפרפרוססינג.

In [ ]:
X_train_full = pd.read_csv('X_train_preprocessed.csv')
y_train_full = pd.read_csv('y_train.csv').squeeze()
X_test       = pd.read_csv('X_test_preprocessed.csv')

print(f'X_train_full : {X_train_full.shape}')
print(f'y_train_full : {y_train_full.shape}')
print(f'X_test       : {X_test.shape}')
print(f'\nFeatures: {X_train_full.columns.tolist()}')

## Step 9.2 — Class Distribution

לפני בניית מודלים, בודקים את התפלגות משתנה המטרה `high_intent`.

אם קיים חוסר איזון, Accuracy לבדו אינו מדד מספיק — נשתמש ב-F1, Recall ו-ROC-AUC.

In [ ]:
print('Class distribution:')
print(y_train_full.value_counts())
print(f'\nPositive class rate : {y_train_full.mean():.1%}')
print(f'Ratio (0:1)         : {(y_train_full==0).sum()} : {(y_train_full==1).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = y_train_full.value_counts().sort_index()
sns.countplot(x=y_train_full, ax=axes[0], palette=['#4C72B0', '#DD8452'])
axes[0].set_title('Class Count', fontsize=12)
axes[0].set_xlabel('high_intent')
axes[0].set_ylabel('Count')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Low Intent (0)', 'High Intent (1)'])
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 20,
                 str(int(bar.get_height())), ha='center', fontsize=11)

axes[1].pie(counts, labels=['Low Intent (0)', 'High Intent (1)'],
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1].set_title('Class Proportion', fontsize=12)

plt.suptitle('Target Variable: high_intent', fontsize=14)
plt.tight_layout()
plt.show()

minority_rate = y_train_full.mean()
if minority_rate < 0.4:
    print(f'\nThe dataset is imbalanced ({minority_rate:.1%} positive class).')
    print('Primary metrics: F1, Recall, ROC-AUC.')

## Step 9.3 — Train / Validation Split

מחלקים את סט האימון ל-80% train ו-20% validation.

`stratify=y` שומר על אותה התפלגות מחלקות בשני הסטים — חשוב במיוחד במקרה של חוסר איזון.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_SEED,
)

print(f'Train set : {X_train.shape}  |  Positive rate: {y_train.mean():.1%}')
print(f'Val set   : {X_val.shape}    |  Positive rate: {y_val.mean():.1%}')
print(f'Test set  : {X_test.shape}')
print('\nstratify=True preserved the class ratio in both splits.')

## Step 9.4 — Candidate Models

מגדירים 5 מודלי סיווג:

| Model | סוג |
|---|---|
| Logistic Regression | מודל לינארי הסתברותי |
| Decision Tree | עץ החלטה — מבצע פיצולים לפי Gini / Information Gain |
| Random Forest | Ensemble של עצים — Majority Vote |
| Naive Bayes | מבוסס משפט בייס עם הנחת אי-תלות |
| SVM | ממקסם Margin בין המחלקות, תומך ב-Kernel non-linear |

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    'Decision Tree'      : DecisionTreeClassifier(random_state=RANDOM_SEED),
    'Random Forest'      : RandomForestClassifier(n_estimators=100,
                                                   random_state=RANDOM_SEED, n_jobs=-1),
    'Naive Bayes'        : GaussianNB(),
    'SVM'                : SVC(probability=True, kernel='rbf', random_state=RANDOM_SEED),
}

print('Candidate models defined:')
for name in models:
    print(f'  - {name}')
print('\nNote: SVM with RBF kernel may take several minutes on large datasets.')

## Step 9.5 — 10-Fold Stratified Cross-Validation

10-fold CV נותן הערכה יציבה יותר מאשר חלוקה חד-פעמית.

מדד הבחירה: F1 ו-ROC-AUC (לא Accuracy בלבד).

In [ ]:
CV_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
cv_results = {}

print('10-Fold Stratified Cross-Validation\n')
for name, model in models.items():
    acc = cross_val_score(model, X_train_full, y_train_full,
                          cv=CV_10, scoring='accuracy', n_jobs=-1)
    f1  = cross_val_score(model, X_train_full, y_train_full,
                          cv=CV_10, scoring='f1', n_jobs=-1)
    auc = cross_val_score(model, X_train_full, y_train_full,
                          cv=CV_10, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = {
        'Accuracy (mean)': round(acc.mean(), 4),
        'Accuracy (std)' : round(acc.std(),  4),
        'F1 (mean)'      : round(f1.mean(),  4),
        'ROC-AUC (mean)' : round(auc.mean(), 4),
    }
    print(f'  {name:<22}  Acc={acc.mean():.3f}±{acc.std():.3f}  '
          f'F1={f1.mean():.3f}  AUC={auc.mean():.3f}')

cv_df = pd.DataFrame(cv_results).T
print()
display(cv_df)

## Step 9.6 — Validation Set Evaluation

מאמנים כל מודל על ה-Train ובודקים על ה-Validation. מחשבים: Accuracy, Precision, Recall, F1, ROC-AUC.

In [ ]:
eval_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else None

    eval_results[name] = {
        'Accuracy' : accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred, zero_division=0),
        'Recall'   : recall_score(y_val, y_pred, zero_division=0),
        'F1'       : f1_score(y_val, y_pred, zero_division=0),
        'ROC-AUC'  : roc_auc_score(y_val, y_proba) if y_proba is not None else float('nan'),
    }

eval_df = pd.DataFrame(eval_results).T.round(4)
print('=== Validation Set Results — All Models ===')
display(eval_df)

In [ ]:
n = len(models)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'])
    ax.set_title(name, fontsize=10)

plt.suptitle('Confusion Matrices — All Models', fontsize=13)
plt.tight_layout()
plt.show()

print('Confusion Matrix Terms:')
print('  TP (True Positive)  — predicted High Intent, actually High Intent')
print('  TN (True Negative)  — predicted Low Intent, actually Low Intent')
print('  FP (False Positive) — predicted High Intent, actually Low Intent  (Type I error)')
print('  FN (False Negative) — predicted Low Intent, actually High Intent  (Type II error)')

## Step 9.7 — ROC Curves

מציגים את עקומות ה-ROC של כל המודלים על אותו גרף. AUC קרוב ל-1 מעיד על מודל טוב.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_val)[:, 1]
        fpr, tpr, _ = roc_curve(y_val, y_proba)
        auc_val = roc_auc_score(y_val, y_proba)
        ax.plot(fpr, tpr, lw=2, label=f'{name}  (AUC = {auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — All Models', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ['Accuracy', 'F1', 'ROC-AUC']):
    vals = eval_df[metric].sort_values(ascending=False)
    bar_colors = ['#4C72B0' if i == 0 else '#aec7e8' for i in range(len(vals))]
    ax.bar(vals.index, vals.values, color=bar_colors, edgecolor='white')
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(ax.patches, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Model Comparison — Accuracy / F1 / ROC-AUC', fontsize=14)
plt.tight_layout()
plt.show()

## Step 9.8 — Decision Tree Visualization

מציגים עץ החלטה עם עומק מוגבל (max_depth=4) לצורך קריאות.

In [ ]:
dt_vis = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED)
dt_vis.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(
    dt_vis,
    feature_names=X_train.columns.tolist(),
    class_names=['Low Intent', 'High Intent'],
    filled=True, rounded=True, fontsize=7, ax=ax,
)
plt.title('Decision Tree (max_depth=4)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Train accuracy (depth=4): {dt_vis.score(X_train, y_train):.3f}')
print(f'Val   accuracy (depth=4): {dt_vis.score(X_val, y_val):.3f}')

## Step 9.9 — Feature Importance (Random Forest)

Random Forest מספק ניקוד חשיבות לכל feature — מאפשר להבין אילו עמודות משפיעות ביותר על התחזית.

In [ ]:
rf_model = models['Random Forest']

feat_imp = (
    pd.Series(rf_model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(13, 5))
feat_imp.head(15).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=13)
ax.set_ylabel('Importance')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(feat_imp.head(10).round(4).to_string())

## Step 9.10 — Hyperparameter Tuning (GridSearchCV)

מכוונים את הפרמטרים של המודל הטוב ביותר לפי F1. `GridSearchCV` בודק שילובים שונים ובוחר את הטוב ביותר.

In [ ]:
best_model_name = eval_df['F1'].idxmax()
print(f'Best model by F1 on validation set: {best_model_name}\n')

param_grids = {
    'Logistic Regression': {
        'C'       : [0.01, 0.1, 1, 10],
        'penalty' : ['l2'],
        'solver'  : ['lbfgs'],
        'max_iter': [1000],
    },
    'Decision Tree': {
        'max_depth'        : [5, 10, 20, None],
        'min_samples_split': [2, 5, 10],
        'criterion'        : ['gini', 'entropy'],
    },
    'Random Forest': {
        'n_estimators'    : [100, 300],
        'max_depth'       : [10, 20, None],
        'min_samples_leaf': [1, 5],
    },
    'Naive Bayes': {},
    'SVM': {
        'C'     : [0.1, 1, 10],
        'kernel': ['rbf', 'linear'],
        'gamma' : ['scale', 'auto'],
    },
}

cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

if param_grids.get(best_model_name):
    base_clf = models[best_model_name]
    grid_search = GridSearchCV(
        base_clf, param_grids[best_model_name],
        cv=cv_tune, scoring='f1', n_jobs=-1, refit=True, verbose=0,
    )
    grid_search.fit(X_train_full, y_train_full)
    best_model_tuned = grid_search.best_estimator_

    print(f'Best hyperparameters for {best_model_name}:')
    for k, v in grid_search.best_params_.items():
        print(f'  {k}: {v}')
    print(f'\nCV F1 before tuning : {eval_df.loc[best_model_name, "F1"]:.4f}')
    print(f'CV F1 after tuning  : {grid_search.best_score_:.4f}')
else:
    best_model_tuned = models[best_model_name]
    best_model_tuned.fit(X_train_full, y_train_full)
    print(f'No grid defined for {best_model_name} — using default parameters.')

## Step 9.11 — Imbalanced Data Handling

משווים שלוש שיטות לטיפול בחוסר האיזון:

| שיטה | תיאור |
|---|---|
| **SMOTE** | יוצר דוגמאות סינתטיות למחלקת המיעוט |
| **ADASYN** | יוצר יותר דוגמאות באזורים קשים לסיווג |
| **RandomUnderSampler** | מקטין את מחלקת הרוב |

השוואה מבוצעת על בסיס Random Forest.

In [ ]:
if not IMBLEARN_AVAILABLE:
    print('Install imbalanced-learn:  pip install imbalanced-learn')
else:
    smote = SMOTE(random_state=RANDOM_SEED)
    X_smote, y_smote = smote.fit_resample(X_train, y_train)

    print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts().sort_index())}')
    print(f'After  SMOTE: {dict(pd.Series(y_smote).value_counts().sort_index())}')

    rf_before = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
    rf_before.fit(X_train, y_train)
    y_pred_before = rf_before.predict(X_val)

    rf_after = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
    rf_after.fit(X_smote, y_smote)
    y_pred_after = rf_after.predict(X_val)

    smote_comparison = pd.DataFrame({
        'Before SMOTE': {
            'Accuracy' : accuracy_score(y_val, y_pred_before),
            'Precision': precision_score(y_val, y_pred_before, zero_division=0),
            'Recall'   : recall_score(y_val, y_pred_before, zero_division=0),
            'F1'       : f1_score(y_val, y_pred_before, zero_division=0),
        },
        'After SMOTE': {
            'Accuracy' : accuracy_score(y_val, y_pred_after),
            'Precision': precision_score(y_val, y_pred_after, zero_division=0),
            'Recall'   : recall_score(y_val, y_pred_after, zero_division=0),
            'F1'       : f1_score(y_val, y_pred_after, zero_division=0),
        },
    }).round(4)

    print('\nRandom Forest — Before vs. After SMOTE:')
    display(smote_comparison)

    fig, ax = plt.subplots(figsize=(8, 5))
    smote_comparison.T.plot(kind='bar', ax=ax,
                             color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.7)
    ax.set_title('Random Forest: Before vs. After SMOTE', fontsize=12)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=0)
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

In [ ]:
if not IMBLEARN_AVAILABLE:
    print('Install imbalanced-learn:  pip install imbalanced-learn')
else:
    sampling_methods = {
        'ADASYN'             : ADASYN(random_state=RANDOM_SEED),
        'RandomUnderSampler' : RandomUnderSampler(random_state=RANDOM_SEED),
    }

    sampling_results = {}
    for method_name, sampler in sampling_methods.items():
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
        rf.fit(X_res, y_res)
        y_pred = rf.predict(X_val)
        sampling_results[method_name] = {
            'Accuracy' : accuracy_score(y_val, y_pred),
            'Precision': precision_score(y_val, y_pred, zero_division=0),
            'Recall'   : recall_score(y_val, y_pred, zero_division=0),
            'F1'       : f1_score(y_val, y_pred, zero_division=0),
        }
        print(f'{method_name}: F1={sampling_results[method_name]["F1"]:.4f}  '
              f'Recall={sampling_results[method_name]["Recall"]:.4f}  '
              f'Samples after resampling: {len(y_res)}')

    sampling_df = pd.DataFrame(sampling_results).T.round(4)
    print()
    display(sampling_df)

## Step 9.12 — Cost-Sensitive Learning

שתי גישות:
1. `class_weight='balanced'` — המודל מעניק משקל גבוה יותר למחלקת המיעוט
2. **שינוי Threshold** — במקום סף ברירת מחדל 0.5, נסה ערכים נמוכים יותר להגדלת Recall

שימושי כאשר FN (החמצת High Intent) חמור יותר מ-FP.

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    random_state=RANDOM_SEED, n_jobs=-1,
)
rf_balanced.fit(X_train, y_train)
y_proba_bal = rf_balanced.predict_proba(X_val)[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6]
threshold_results = {}

for t in thresholds:
    y_pred_t = (y_proba_bal >= t).astype(int)
    threshold_results[f'Threshold={t}'] = {
        'Precision': precision_score(y_val, y_pred_t, zero_division=0),
        'Recall'   : recall_score(y_val, y_pred_t, zero_division=0),
        'F1'       : f1_score(y_val, y_pred_t, zero_division=0),
        'Accuracy' : accuracy_score(y_val, y_pred_t),
    }

threshold_df = pd.DataFrame(threshold_results).T.round(4)
print('Cost-Sensitive Learning — Threshold Analysis (class_weight=balanced):')
display(threshold_df)

print('\nהורדת ה-Threshold מגדילה את ה-Recall (פחות החמצות) אך מקטינה את ה-Precision.')
print('הבחירה תלויה בעלות העסקית של FN לעומת FP.')

## Step 9.13 — Precision-Recall Curve

בנתונים עם חוסר איזון, עקומת PR לעתים קרובות אינפורמטיבית יותר מ-ROC. הקו האדום מייצג מודל ללא מיומנות (No-Skill baseline).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_val)[:, 1]
        precision_vals, recall_vals, _ = precision_recall_curve(y_val, y_proba)
        ap = average_precision_score(y_val, y_proba)
        ax.plot(recall_vals, precision_vals, lw=2, label=f'{name}  (AP = {ap:.3f})')

no_skill = y_val.mean()
ax.axhline(no_skill, color='red', linestyle='--', lw=1,
           label=f'No-skill  (P = {no_skill:.2f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curves — All Models', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

## Step 9.14 — Final Model Evaluation and Test Predictions

מאמנים את המודל המכוון על כל נתוני האימון ומפיקים תחזיות על סט הטסט.

In [ ]:
print(f'Selected model: {best_model_name} (tuned)\n')

best_model_tuned.fit(X_train, y_train)
y_pred_final  = best_model_tuned.predict(X_val)
y_proba_final = (
    best_model_tuned.predict_proba(X_val)[:, 1]
    if hasattr(best_model_tuned, 'predict_proba') else None
)

print('=== Classification Report ===')
print(classification_report(y_val, y_pred_final,
                             target_names=['Low Intent (0)', 'High Intent (1)']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_val, y_pred_final,
    display_labels=['Low Intent', 'High Intent'],
    cmap='Blues', ax=axes[0],
)
axes[0].set_title(f'Confusion Matrix — {best_model_name}', fontsize=12)

if y_proba_final is not None:
    fpr, tpr, _ = roc_curve(y_val, y_proba_final)
    auc_val = roc_auc_score(y_val, y_proba_final)
    axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_val:.3f}')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve — Final Model', fontsize=12)
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_model_tuned.fit(X_train_full, y_train_full)

y_pred_test  = best_model_tuned.predict(X_test)
y_proba_test = (
    best_model_tuned.predict_proba(X_test)[:, 1]
    if hasattr(best_model_tuned, 'predict_proba')
    else np.full(len(X_test), float('nan'))
)

predictions_df = pd.DataFrame({
    'predicted_high_intent': y_pred_test,
    'proba_high_intent'    : y_proba_test.round(4),
})
predictions_df.to_csv('test_predictions.csv', index=False)

print(f'Saved: test_predictions.csv  ({len(predictions_df)} rows)')
print(f'\nPredicted class distribution:')
print(predictions_df['predicted_high_intent'].value_counts().to_string())
print(f'\nMean predicted probability: {y_proba_test.mean():.3f}')
display(predictions_df.head(10))